In [2]:
from bitarray import bitarray
import hashlib
from hashlib import sha3_256, sha256, blake2b
import math 
import mmh3
import string
import json
import requests

In [38]:
# You can read the list one at a time using the code snippet below

words = []

with open('words.txt') as f:
    for line in f:
        word = line.strip().lower()
        words.append(word)

# words_set = set(words)

In [40]:
from hashlib import sha3_256, sha256, blake2b
size = 10_000_000
def my_hash(s):
    return int(sha256(s.lower().encode()).hexdigest(), 16) % size # given

def my_hash2(s):
    return int(blake2b(s.lower().encode()).hexdigest(), 16) % size

def my_hash3(s):
    return int(sha3_256(s.lower().encode()).hexdigest(), 16) % size

three_hash_functions = [my_hash, my_hash2, my_hash3] # set functions that will later dictate how many hashes are used
two_hash_functions = [my_hash, my_hash2]
one_hash_function = [my_hash]

In [41]:
from bitarray import bitarray

class BloomFilter:
    def __init__(self, size, hash_functions):
        self.size = size
        self.hash_functions = hash_functions
        self.bit_array = bitarray(self.size)
        self.bit_array.setall(0)

    def add(self, item):
        for func in self.hash_functions:
            index = func(item)
            if index >= self.size: # sanity check for index and hash size
                raise ValueError(f"Index {index} out of bounds for size {self.size}")
            self.bit_array[index] = 1

    def check(self, item):
        for func in self.hash_functions:
            index = func(item)
            if self.bit_array[index] == 0:
                return False
        return True


In [42]:
#  For each word in the list, apply all three hash functions below and set the corresponding bits in the bitarray. These hash functions all return integers in [0, size), where size is some integer specified elsewhere

bloomf_3 = BloomFilter(size,three_hash_functions) # added words to bloom filter using sizen given and all three hashed
bloomf_2 = BloomFilter(size, two_hash_functions) # two hashes used
bloomf_1 = BloomFilter(size, one_hash_function) # one hash used
bloom_test = BloomFilter(size, three_hash_functions)

for word in words: # added words to each type of bloom filter
    bloomf_3.add(word)
    bloomf_2.add(word)
    bloomf_1.add(word)

bloom_test.add("apple")
bloom_test.add("banana")

In [43]:
bloom_test.check("apple")

True

In [44]:
#b. Create a function that checks all possible single-character substitutions for a given word using the Bloom filter. . 

#I need the function to take in each word, replace a single character, compare to the rest of list and return if it is a match. Repeat for all letter combinations

#Asked ChatGPT how to make a function that replces single character in given word
def single_char_changes(word): 
    replaced_words = []
    for i in range(len(word)):
        for letter in string.ascii_lowercase:
            if word[i] != letter:
                changed = word[:i] + letter + word[i+1:]
                replaced_words.append(changed)
    return replaced_words

#Tested that the function does single character substitution for word given
len(single_char_changes('cat'))

75

In [45]:
#create full function
#Asked ChatGPT how to compare again the bloom filter and ensure function is doing what i want. 
def spell_check(word, bloom):
    candidates = single_char_changes(word)
    for candidate in candidates:
        if bloom.check(candidate): #check  if hash function is there 
            print(f"{candidate} is a Match!")


In [46]:
spell_check('bpple', bloom_test)

apple is a Match!


In [48]:
spell_check('floeer', bloomf_1)

bloeer is a Match!
qloeer is a Match!
fyoeer is a Match!
flofer is a Match!
floter is a Match!
flower is a Match!
floeqr is a Match!
floees is a Match!


In [49]:
spell_check('floeer', bloomf_2)

fyoeer is a Match!
floter is a Match!
flower is a Match!


In [47]:
spell_check('floeer', bloomf_3)

floter is a Match!
flower is a Match!


In [50]:
#call in typos and visualize
with open('typos.json', 'r') as file2:
    typos = json.load(file2)

In [51]:
print(typos[:5])

[['soprabi', 'soprani'], ['rosan', 'rosan'], ['jeresiologer', 'heresiologer'], ['wrinkvy', 'wrinkly'], ['seaweeds', 'seaweeds']]
